In [1]:
import os, requests, json, pyodbc, logging, time
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv
from sqlalchemy import create_engine
from tqdm import tqdm
from datetime import datetime, timedelta
import unicodedata
import numpy as np


load_dotenv()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

API_KEY = os.getenv("INAGENT_API_KEY")
ENDPOINT = os.getenv("INAGENT_URL")

CREW_MAPPING = {
    os.getenv("INAGENT_CREW_ID"): "MEDICA",
    os.getenv("INAGENT_CREW_ID2"): "OMV"
}
CREW_IDS = list(CREW_MAPPING.keys())


In [2]:
hoy = datetime.now()
ayer = hoy - timedelta(days=1)

env_start = os.getenv("START_DATE")
env_end = os.getenv("END_DATE")

if env_start:
    start_date = env_start
else:
    start_date = ayer.strftime("%Y-%m-%dT00:00:00")

if env_end:
    end_date = env_end
else:
    end_date = ayer.strftime("%Y-%m-%dT23:59:59")

print(f"Extrayendo datos desde {start_date} hasta {end_date}")


Extrayendo datos desde 2026-03-20T00:00:00 hasta 2026-04-04T23:59:59


In [3]:
def to_unix_ms(iso_date):
    dt = datetime.fromisoformat(iso_date).replace(tzinfo=timezone.utc)
    return int(dt.timestamp() * 1000)

def safe_json_parse(val):
    try:
        return json.loads(val) if (val and val != 'null') else {}
    except:
        return {}

print(" Herramientas listas: to_unix_ms y safe_json_parse.")

 Herramientas listas: to_unix_ms y safe_json_parse.


In [4]:
all_data = []
page_size = 100 # Límite recomendado por la API [cite: 141]

for crew in CREW_IDS:
    page = 0
    label_origen = CREW_MAPPING.get(crew, "DESCONOCIDO")
    logger.info(f"Extrayendo datos de {label_origen} (ID: {crew})")
    
    while True:
        params = {
            "crew_id": crew,
            "start_ts": to_unix_ms(start_date),
            "end_ts": to_unix_ms(end_date),
            "page": page,
            "pageSize": page_size
        }
        
        headers = {"apikey": API_KEY}
        res = requests.get(ENDPOINT, headers=headers, params=params)
        
        if res.status_code != 200:
            logger.error(f"Fallo en {label_origen}, Página {page}: {res.text}")
            break
            
        data_payload = res.json().get("data", {})
        rows = data_payload.get("rows", [])
        
        if page == 0 and crew == CREW_IDS[0]:
            cols = data_payload.get("dataSchema", {}).get("columnNames", [])
            if "Origen" not in cols:
                cols.append("Origen")
        
        if not rows:
            break
            
        for row in rows:
            row.append(label_origen)
            
        all_data.extend(rows)
        logger.info(f"Página {page} de {label_origen} lista. Total: {len(all_data)}")
        
        if len(rows) < page_size: # Fin de datos para este equipo [cite: 262]
            break
        page += 1

df_raw = pd.DataFrame(all_data, columns=cols)


2026-04-05 19:01:24,365 - INFO - Extrayendo datos de MEDICA (ID: 766cddc2-eaf9-464e-8f6d-8854ef927ff3)
2026-04-05 19:01:25,380 - INFO - Página 0 de MEDICA lista. Total: 14
2026-04-05 19:01:25,380 - INFO - Extrayendo datos de OMV (ID: 2539ab63-c408-446a-b13e-1eef4f7c1ba3)
2026-04-05 19:01:26,149 - INFO - Página 0 de OMV lista. Total: 29


In [5]:
native_whitelist = [
    'Id', 
    'Id Externo',
    'Id Canal',
    'Canal',
    'Timestamp',
    'Inicio',
    'Fin',
    'Duración (s)', 
    'Análisis Sentimental',
    'Tema general de la conversación',
    #'Resumen',
    'Fue resuelta',
    'Fue solo agradecimiento',
    'Herramientas Usadas',
    'Es saliente',
    'Fue abandonada',
    'Contexto',
    'Origen'
]

In [6]:
context_whitelist = [
    'toolLogs'
]

In [7]:
df_native = df_raw[[c for c in native_whitelist if c in df_raw.columns]].copy()

ctx_raw = pd.json_normalize(df_raw['Contexto'].apply(safe_json_parse))
ctx_raw.columns = [c.replace(".", "_") for c in ctx_raw.columns]

In [8]:
ctx_selected = ctx_raw[[c for c in context_whitelist if c in ctx_raw.columns]].add_prefix('ctx_')
df_base = pd.concat([df_native, ctx_selected], axis=1)
print(f"{df_base.shape}")

(29, 18)


In [10]:
col_logs = 'ctx_toolLogs'

df_tools_exploded = df_base[['Id', col_logs]].dropna(subset=[col_logs]).explode(col_logs)
tool_rows = df_tools_exploded[col_logs].apply(lambda x: x if isinstance(x, (dict, list)) else safe_json_parse(x)).tolist()
df_tools_flat = pd.json_normalize(tool_rows)

In [11]:
tool_whitelist = [
    'URL_fetch', 'body_fetch', 'return_fetch', 'tool', 'status', 
    'timestamp', 'code_fetch',
    'return_fetch.message','return_fetch.status','return_fetch.success',
    'return_fetch.data.policy_number','return_fetch.data.program_name',
    'return_fetch.data.client_card','body_fetch.cas','body_fetch.cancelMotive',
    'body_fetch.idBeneficiary','body_fetch.scheduleDate','body_fetch.specialty',
    'return_fetch.response.selected_dentist','return_fetch.response.idDoctor',
    'return_fetch.response.Kinship','return_fetch.response.nameDoctor','return_fetch.response.costPIFDoctor',
    'return_fetch.response.typeOfService','return_fetch.response.name_service',
]

In [ ]:
cols_t = [c for c in tool_whitelist if c in df_tools_flat.columns]
df_tools_final = df_tools_flat[cols_t].copy()
df_tools_final.columns = [f"tool_{c.replace('.', '_')}" for c in df_tools_final.columns]

df_tools_final.index = df_tools_exploded.index
df_tools_merged = pd.concat([df_tools_exploded[['Id']], df_tools_final], axis=1)

print(f'{df_tools_merged.shape}')

In [ ]:
df_final = df_base.merge(df_tools_merged, on='Id', how='left')
df_final.columns = [c.replace(" ", "_").replace("(", "").replace(")", "").replace(".", "_") for c in df_final.columns]

In [ ]:
def aplicar_ancla_maestra(df): 
    df['tool_timestamp_dt'] = pd.to_datetime(df['tool_timestamp'], errors='coerce')
    df = df.sort_values(by=['Id', 'tool_timestamp_dt'], ascending=[True, False])
    
    es_el_ancla = ~df.duplicated(subset=['Id'], keep='first')
    
    df['Indicador_Sesion'] = np.where(es_el_ancla, 1.0, 0.0)
    df['Estatus_Final'] = np.where(es_el_ancla, df['tool_tool'], None)
    
    return df

df_preparado = aplicar_ancla_maestra(df_final)

In [ ]:
df_preparado.info()

In [ ]:
def crear_id_compuesto_pro(df):
    logger.info(" Generando identificadores únicos...")
    

    df['tool_tool'] = df['tool_tool'].fillna('SIN_HERRAMIENTA') # <--- CLAVE
    
    tool = df['tool_timestamp'].astype(str)
    
    df['id_registro'] = (
        df['Id'].astype(str) + "_" + 
        df['tool_tool'].astype(str) + "_" + 
        tool
    )
    return df

In [ ]:
crear_id_compuesto_pro(df_preparado)

In [ ]:
columnas_sql_reales = [
    'id_registro','Id', 'tool_timestamp','Origen','Estatus_Final','Indicador_Sesion',
    
    'Id_Externo', 'Id_Canal', 'Canal', 'Timestamp', 
    'Inicio', 'Fin', 'Duracion_s', 'Análisis_Sentimental', 
    'Tema_general_de_la_conversación', 'Fue_resuelta', 
    'Fue_solo_agradecimiento', 'Herramientas_Usadas', 
    'Es_saliente', 'Fue_abandonada',
    
    'tool_URL_fetch', 'tool_body_fetch', 
    'tool_return_fetch', 'tool_tool', 'tool_status', 
    'tool_code_fetch', 'tool_return_fetch_message', 
    'tool_return_fetch_status', 'tool_return_fetch_success',
    
    'tool_return_fetch_data_policy_number', 
    'tool_return_fetch_data_program_name', 
    'tool_return_fetch_data_client_card',
    
    'tool_body_fetch_cas', 'tool_body_fetch_cancelMotive', 
    'tool_body_fetch_idBeneficiary', 'tool_body_fetch_scheduleDate', 
    'tool_body_fetch_specialty',
    
    'tool_return_fetch_response_selected_dentist',
    'tool_return_fetch_response_idDoctor',
    'tool_return_fetch_response_Kinship',
    'tool_return_fetch_response_nameDoctor',
    'tool_return_fetch_response_costPIFDoctor',
    'tool_return_fetch_response_typeOfService',
    'tool_return_fetch_response_name_service'
]

In [ ]:
def pipeline_maestro_final(df, whitelist):
    df_sql = df.copy()
    
    # 1. Limpieza de nombres igual que antes
    def limpiar_nombres(txt):
        if not isinstance(txt, str): return txt
        txt = "".join(c for c in unicodedata.normalize('NFD', txt) if unicodedata.category(c) != 'Mn')
        return txt.replace(" ", "_").replace("(", "").replace(")", "").replace(".", "_")

    df_sql.columns = [limpiar_nombres(c) for c in df_sql.columns]
    whitelist_limpia = [limpiar_nombres(c) for c in whitelist]

    # 2. Mapeos Estratégicos (Corregido sin espacios)
    if 'Origen' in df_sql.columns:
        df_sql['Id_Canal'] = df_sql['Origen']
        df_sql['Id_Externo'] = df_sql['id_registro'] 
        df_sql['tool_return_fetch_response_costPIFDoctor'] = df_sql['Indicador_Sesion']
        df_sql['tool_return_fetch_response_typeOfService'] = df_sql['Estatus_Final']

    # 3. Filtrado por whitelist
    df_sql = df_sql[[c for c in whitelist_limpia if c in df_sql.columns]]

    # 4. LISTA MAESTRA DE COLUMNAS NUMÉRICAS (¡Aquí estaba el error!)
    # Debemos incluir todas las que en tu tabla son float o int
    cols_num = [
        'Duracion_s', 'Herramientas_Usadas', 'tool_code_fetch',
        'tool_return_fetch_httpCode', 'tool_return_fetch_idAppointment',
        'tool_body_fetch_lng_base', 'tool_body_fetch_secondIdService',
        'tool_body_fetch_firstIdService', 'tool_return_fetch_response_selected_dentist',
        'tool_return_fetch_response_costPIFDoctor', 'Cantidad_de_preguntas_a_QNA'
    ]

    # 5. CONVERSIÓN SEGURA
    for col in df_sql.columns:
        if col in cols_num:
            # Forzamos conversión a número y reemplazamos NaN por None (NULL en SQL)
            df_sql[col] = pd.to_numeric(df_sql[col], errors='coerce')
            # Es vital usar .replace({np.nan: None}) para que pyodbc no envíe "nan" como texto
            df_sql[col] = df_sql[col].astype(object).where(pd.notnull(df_sql[col]), None)
            
        elif any(x in col for x in ['Fue_', 'Es_', 'Cerrada_']):
            # BIT/Booleanos a 0 o 1
            df_sql[col] = pd.to_numeric(df_sql[col], errors='coerce').fillna(0).astype(int)
            
        else:
            # TEXTO: Evitamos que 'nan' se guarde como string
            df_sql[col] = df_sql[col].astype(str).replace(['nan', 'None', 'NaN', 'null'], None)
            df_sql[col] = df_sql[col].where(df_sql[col].notnull(), None)

    return df_sql

In [ ]:
df_listo = pipeline_maestro_final(df_preparado, columnas_sql_reales)

In [ ]:
para_SQL = [
    #'id_registro',
    'Id', 'tool_timestamp',
    
    'Id_Externo', 'Id_Canal', 'Canal', 'Timestamp', 
    'Inicio', 'Fin', 'Duracion_s', 'Analisis_Sentimental', 
    'Tema_general_de_la_conversacion', 'Fue_resuelta', 
    'Fue_solo_agradecimiento', 'Herramientas_Usadas', 
    'Es_saliente', 'Fue_abandonada',
    
    'tool_URL_fetch', 'tool_body_fetch', 
    'tool_return_fetch', 'tool_tool', 'tool_status', 
    'tool_code_fetch', 'tool_return_fetch_message', 
    'tool_return_fetch_status', 'tool_return_fetch_success',
    
    'tool_return_fetch_data_policy_number', 
    'tool_return_fetch_data_program_name', 
    'tool_return_fetch_data_client_card',
    
    'tool_body_fetch_cas', 'tool_body_fetch_cancelMotive', 
    'tool_body_fetch_idBeneficiary', 'tool_body_fetch_scheduleDate', 
    'tool_body_fetch_specialty',
    
    'tool_return_fetch_response_selected_dentist',
    'tool_return_fetch_response_idDoctor',
    'tool_return_fetch_response_Kinship',
    'tool_return_fetch_response_nameDoctor',
    'tool_return_fetch_response_costPIFDoctor',
    'tool_return_fetch_response_typeOfService',
    'tool_return_fetch_response_name_service'
]

df_produccion = df_listo[[c for c in para_SQL if c in df_listo.columns]].copy()

In [ ]:
df_produccion.to_csv("Produccion_sql.csv",index=False)

In [ ]:
df_para_sql = df_produccion.copy()

TABLE_NAME = "dbo.inagent" #

conn_str = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')},{os.getenv('DB_PORT')};"
    f"DATABASE={os.getenv('BD')};"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASS')}"
)

try:
    conn = pyodbc.connect(conn_str)
    cursor = conn.cursor()
    cursor.fast_executemany = True 

    cursor.execute(f"IF OBJECT_ID('tempdb..#stg_inagent') IS NOT NULL DROP TABLE #stg_inagent")
    
    cols = df_para_sql.columns.tolist()
    col_names_bracketed = ", ".join(f"[{c}]" for c in cols)
    
    cursor.execute(f"SELECT TOP 0 {col_names_bracketed} INTO #stg_inagent FROM {TABLE_NAME}") 

    placeholders = ", ".join("?" for _ in cols)
    sql_insert = f"INSERT INTO #stg_inagent ({col_names_bracketed}) VALUES ({placeholders})"
    
    data_to_load = [tuple(x) for x in df_para_sql.values]
    
    logger.info(f" Subiendo {len(data_to_load)} registros a Staging...")
    cursor.executemany(sql_insert, data_to_load)
    
    sql_merge = f"""
    MERGE {TABLE_NAME} AS target
    USING #stg_inagent AS source
    ON (target.Id_Externo= source.Id_Externo)
    WHEN MATCHED THEN
        UPDATE SET 
            target.Analisis_Sentimental = source.Analisis_Sentimental,
            target.Tema_general_de_la_conversacion = source.Tema_general_de_la_conversacion,
            target.tool_status = source.tool_status,
            target.tool_return_fetch_message = source.tool_return_fetch_message
    WHEN NOT MATCHED THEN
        INSERT ({col_names_bracketed})
        VALUES ({', '.join(f'source.[{c}]' for c in cols)});
    """
    
    logger.info("Ejecutando MERGE en tabla definitiva...")
    cursor.execute(sql_merge)
    conn.commit()
    logger.info(f"ÉXITO: {len(df_para_sql)} registros sincronizados correctamente.")

except Exception as e:
    if 'conn' in locals(): conn.rollback()
    logger.error(f"Error en SQL: {e}")
finally:
    if 'cursor' in locals(): cursor.close()
    if 'conn' in locals(): conn.close()

In [ ]:
tools_master_whitelist = [
    'inicializar_sesion',           # Obtiene información del cliente [cite: 47]
    'CerrarConversacionPorUsuario', # Cierre por decisión del usuario [cite: 48]
    'CerrarSesionPorTimeout',       # Cierre por inactividad [cite: 49]
    'get_fecha',                    # Obtiene fecha actual [cite: 50]
    'upsert_beneficiario',          # Registra/actualiza beneficiario [cite: 51]
    'get_beneficiarios',            # Obtiene lista de beneficiarios [cite: 52]
    'get_servicios_siniestralidad', # Consulta de servicios [cite: 53]
    'validar_tarjeta',              # Consulta beneficios [cite: 54]
    'TransferenciaAsesor',          # Envío a agente humano [cite: 113] 
    'get_titular',                  # Obtiene ID del titular [cite: 55]
    'get_horarios',                 # Consulta disponibilidad [cite: 56]
    'cancelar_cita',                # Cancela cita agendada [cite: 57]
    'set_checkup',                  # Crea nueva cita [cite: 58]
    'set_context'                   # Crea contexto con Talkdesk [cite: 59]
]